# STEP 4: Build subscription spells


In [1]:
## 1. setup
import os
from pathlib import Path
import duckdb
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
print("Working in:", Path.cwd())

con = duckdb.connect("kkbox.duckdb")
con.sql("SET memory_limit='6GB'")          # about half your RAM is a safe choice
con.sql("SET temp_directory='data/tmp'")   # where DuckDB spills if memory runs out

def run_sql_file(path):
    """Run every statement in a .sql file."""
    for stmt in Path(path).read_text(encoding="utf-8").split(";"):
        if stmt.strip():
            con.sql(stmt)
    print("Done:", path)

Working in: d:\0statistic project\subscription-lifecycle-churn-analysis


In [2]:
## 2.Check bad expiry dates
con.sql("""
    SELECT is_cancel,
           COUNT(*)                                                         AS n,
           COUNT(*) FILTER (WHERE expire_date = DATE '1970-01-01')          AS expire_1970,
           COUNT(*) FILTER (WHERE expire_date > tx_date + INTERVAL 450 DAY) AS expire_far_future
    FROM tx_clean GROUP BY 1
""").df()

,is_cancel,n,expire_1970,expire_far_future
0,0,22077552,0,130372
1,1,891402,246,2873


In [4]:
## 3. spell SQL
run_sql_file("sql/03_spells.sql")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Done: sql/03_spells.sql


In [5]:
### 4. Integrity tests
tests = {
    "Transactions not assigned to exactly one spell":
        "SELECT (SELECT COUNT(*) FROM tx_for_spells) - (SELECT SUM(n_transactions) FROM spells_final)",
    "Spells with negative duration":
        "SELECT COUNT(*) FROM spells_final WHERE duration_days < 0",
    "Spells starting within 30 days of the previous spell's end":
        """SELECT COUNT(*) FROM (
               SELECT spell_start,
                      LAG(spell_end) OVER (PARTITION BY msno ORDER BY spell_id) AS prev_end
               FROM spells_final)
           WHERE prev_end IS NOT NULL AND spell_start <= prev_end + INTERVAL 30 DAY""",
    "Users whose spell ids are not 1, 2, 3, ...":
        """SELECT COUNT(*) FROM (
               SELECT msno, MAX(spell_id) AS m, COUNT(*) AS c FROM spells_final GROUP BY msno)
           WHERE m <> c""",
}
for name, q in tests.items():
    print(f"{con.sql(q).fetchone()[0]:>8,}  {name}")

       0  Transactions not assigned to exactly one spell
  13,541  Spells with negative duration
       0  Spells starting within 30 days of the previous spell's end
       0  Users whose spell ids are not 1, 2, 3, ...


In [6]:
### 5. Check consistency with KKBox’s labels
con.sql("""
    WITH last_spell AS (
        SELECT msno, arg_max(spell_end, spell_id) AS last_end
        FROM spells_final GROUP BY msno
    )
    SELECT (s.last_end > DATE '2017-04-30') AS covered_beyond_april,
           l.is_churn,
           COUNT(*) AS n
    FROM labels l
    LEFT JOIN last_spell s USING (msno)
    GROUP BY ALL ORDER BY ALL
""").df()

,covered_beyond_april,is_churn,n
0,False,0,806689
1,False,1,61496
2,True,0,76940
3,True,1,25719
4,<NA>,0,1
5,<NA>,1,115


In [ ]:
### 6a. Look at the result
con.sql("""
    SELECT COUNT(*)                                  AS n_spells,
           COUNT(DISTINCT msno)                      AS n_users,
           ROUND(COUNT(*) / COUNT(DISTINCT msno), 2) AS spells_per_user,
           ROUND(100 * AVG(churned::INT), 1)         AS pct_churned,
           ROUND(100 * AVG(is_reactivation::INT), 1) AS pct_reactivations,
           ROUND(100 * AVG(left_truncated::INT), 1)  AS pct_left_truncated,
           MEDIAN(duration_days)                     AS median_duration_days
    FROM spells_final
""").df()

,n_spells,n_users,spells_per_user,pct_churned,pct_reactivations,pct_left_truncated,median_duration_days
0,3030066,2424501,1.25,59.6,20.0,18.0,128.0


In [8]:
### 6b. Look at the result
user = con.sql("SELECT msno FROM spells_final WHERE spell_id = 3 LIMIT 1").fetchone()[0]
display(con.sql(f"""SELECT tx_date, expire_date, is_cancel, payment_plan_days
                    FROM tx_for_spells WHERE msno = '{user}' ORDER BY tx_date""").df())
display(con.sql(f"""SELECT spell_id, spell_start, spell_end, n_transactions, churned
                    FROM spells_final WHERE msno = '{user}' ORDER BY spell_id""").df())

,tx_date,expire_date,is_cancel,payment_plan_days
0,2015-01-02,2015-02-03,0,30
1,2015-01-27,2015-01-28,1,30
2,2015-01-30,2015-02-28,0,31
3,2015-03-01,2015-03-31,0,31
4,2015-04-01,2015-04-30,0,31
5,2015-05-01,2015-06-01,0,0
6,2015-07-01,2015-08-01,0,30
7,2015-08-01,2015-09-01,0,30
8,2015-09-01,2015-10-01,0,30
9,2015-10-01,2015-11-01,0,30


,spell_id,spell_start,spell_end,n_transactions,churned
0,1.0,2015-01-02,2016-01-01,12,True
1,2.0,2016-02-01,2016-03-01,1,True
2,3.0,2016-04-07,2016-05-06,1,True
3,4.0,2016-08-01,2017-04-06,8,False


## Summary: subscription spells

**Input.** Excluded 133,491 transactions (0.58% of `tx_clean`) with placeholder (1970-01-01) or far-future expiry dates (more than 450 days after the transaction): 130,372 payments and 3,119 cancellations. Some far-future expiries may be legitimate (e.g. a long plan bought while time was left on the old one), but at 0.6% this does not affect the results.

**Spells.** Built 3,030,066 spells for 2,424,501 users (1.25 spells per user). A spell ends when no new transaction follows within 30 days of the membership expiring.
- 59.6% of spells ended in churn; 40.4% are still active at the data end (31 March 2017) and are treated as censored.
- 20.0% are reactivations (a user's second or later spell).
- 18.0% are left-truncated (start in January 2015 and may have begun earlier); these are excluded from tenure analysis in Step 6.
- Median spell duration: 128 days.

The 59.6% is not comparable to the 9% churn rate in the labels: 9% is churn at a single renewal (March 2017), while 59.6% is the share of spells that ended at any point over 27 months.

**Integrity tests.** [All four tests return 0. / Note any exceptions and how they are handled.]

**Example check.** A user with four spells confirms the logic: breaks occur exactly where a gap after expiry exceeds 30 days (in one case by a single day), and the last spell is correctly censored. A 0-day plan in this history behaves like a normal monthly payment, supporting the decision to keep 0-day plans.

**Consistency with KKBox's labels (open issue).** Among labelled users whose last spell already runs beyond April 2017, 25.1% (25,719 of 102,659) are labelled as churners, compared with 7.1% for all other labelled users. The expectation was close to 0%, so my spell logic and KKBox's labelling rule differ for these users. Possible causes under investigation: a different treatment of cancellations in KKBox's labeller, events in April 2017 not visible in the data (e.g. refunds after early renewal), and same-day ordering of payments and cancellations. 116 labelled users have no spells because all their transactions had invalid expiry dates.

**Next.** The spells are internally consistent and are used as-is for cohorts and survival analysis (Steps 5–6). The label discrepancy is resolved before the labels are used in Step 8. A robustness check with 14- and 60-day windows is planned.